<center><h1 style = "background:#000000 ;color:white;border:0;font-weight:bold">Random Forest</h1></center>

Random Forest is an **ensemble** of decision trees trained on bootstrapped samples with random feature subsets — reducing variance while keeping low bias.

### Topics:
1. [Theory — Bagging & Random Subspaces](#1)
2. [Dataset & Visualization](#2)
3. [n_estimators vs Accuracy](#3)
4. [Feature Importance](#4)
5. [Evaluation of Model](#5)

In [ ]:
import seaborn as sns
rf_colors = ['#1a472a', '#2a9d8f', '#57cc99', '#c7f9cc', '#0d2b1a']
print("Random Forest Colors:")
sns.palplot(sns.color_palette(rf_colors))

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns; sns.set()
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score)
from sklearn.inspection import permutation_importance
import warnings; warnings.filterwarnings("ignore")
%matplotlib inline
np.random.seed(42)
print("Libraries loaded.")

## Let's Start!

<a id = "1"></a>
<center><h1 style = "background:#000000 ;color:white;border:0;font-weight:bold">Theory — Bagging & Random Subspaces</h1></center>

## Bootstrap Aggregation (Bagging)

Each tree in the forest is trained on a **bootstrap sample** — n rows drawn with replacement. On average, ~63.2% of rows are included; the rest are **Out-of-Bag (OOB)**.

$$P(\text{row not selected}) = \left(1 - \frac{1}{n}\right)^n \xrightarrow{n\to\infty} e^{-1} \approx 0.368$$

## Random Feature Subsets

At each split, only $\sqrt{p}$ features are considered (classification). This **decorrelates** the trees so their errors cancel when averaged.

## Prediction

- Classification: **majority vote** across all trees
- Regression: **mean** of all tree predictions

<SCRIPT SRC='https://cdn.mathjax.org/mathjax/latest/MathJax.js?config=TeX-AMS-MML_HTMLorMML'></SCRIPT>
<SCRIPT>MathJax.Hub.Config({ tex2jax: {inlineMath: [['$','$'],['\\(','\\)']]}});</SCRIPT>

<a id = "2"></a>
<center><h1 style = "background:#1a472a ;color:white;border:0;font-weight:bold">Information About Dataset</h1></center>

**Breast Cancer Wisconsin** — 569 samples, 30 features, binary target.

In [ ]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
df = X.copy(); df["target"] = y
print(f"Shape: {df.shape}")
print(df["target"].value_counts().rename({0:"malignant",1:"benign"}))
df.describe().T.head(8)

<center><h1 style = "background:#2a9d8f ;color:white;border:0;font-weight:bold">Data Visualization</h1></center>

In [ ]:
top_features = ["mean radius", "mean perimeter", "mean area", "mean concavity"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8), dpi=100)

for ax, feat in zip(axes.ravel(), top_features):
    for label, color in [(0, "#1a472a"), (1, "#57cc99")]:
        sns.kdeplot(df[df["target"]==label][feat], ax=ax,
                    label=data.target_names[label], color=color, fill=True, alpha=0.4)
    ax.set_title(feat); ax.legend()

plt.suptitle("Feature Distributions by Class", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(14, 8), dpi=100)
sns.heatmap(X.iloc[:, :10].assign(target=y).corr(numeric_only=True),
            annot=True, fmt=".2f", cmap="Greens", linewidths=0.5)
plt.title("Correlation Heatmap (first 10 features)")
plt.show()

<center><h1 style = "background:#57cc99 ;color:black;border:0;font-weight:bold">Train-Test Split</h1></center>

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

print(f"Train: {X_train.shape}  Test: {X_test.shape}")
print(f"Positive rate — train: {y_train.mean():.3f}  test: {y_test.mean():.3f}")

<a id = "3"></a>
<center><h1 style = "background:#c7f9cc ;color:black;border:0;font-weight:bold">n_estimators vs Accuracy</h1></center>

In [ ]:
n_tree_range = [1, 5, 10, 25, 50, 100, 200, 300]
oob_scores, test_scores = [], []

for n_est in n_tree_range:
    rf = RandomForestClassifier(n_estimators=n_est, oob_score=True,
                                random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    oob_scores.append(rf.oob_score_)
    test_scores.append(accuracy_score(y_test, rf.predict(X_test)))

plt.figure(figsize=(10, 5), dpi=100)
plt.plot(n_tree_range, oob_scores,  "o-", label="OOB accuracy",  color="#1a472a")
plt.plot(n_tree_range, test_scores, "s-", label="Test accuracy", color="#2a9d8f")
plt.xlabel("Number of trees")
plt.ylabel("Accuracy")
plt.title("Random Forest: n_estimators vs Accuracy", fontweight="bold")
plt.legend()
plt.show()
print(f"Best test accuracy: {max(test_scores):.4f} at n_estimators={n_tree_range[np.argmax(test_scores)]}")

<center><h1 style = "background:#2a9d8f ;color:white;border:0;font-weight:bold">Random Forest Model</h1></center>

In [ ]:
best_rf = RandomForestClassifier(n_estimators=200, oob_score=True,
                                  random_state=42, n_jobs=-1)
best_rf.fit(X_train, y_train)
pred = best_rf.predict(X_test)
prob = best_rf.predict_proba(X_test)[:, 1]

print(f"OOB Score    : {best_rf.oob_score_:.4f}")
print(f"Train Acc    : {best_rf.score(X_train, y_train):.4f}")
print(f"Test Acc     : {accuracy_score(y_test, pred):.4f}")
print(f"ROC-AUC      : {roc_auc_score(y_test, prob):.4f}")

<a id = "4"></a>
<center><h1 style = "background:#1a472a ;color:white;border:0;font-weight:bold">Feature Importance</h1></center>

In [ ]:
mdi  = best_rf.feature_importances_
perm = permutation_importance(best_rf, X_test, y_test, n_repeats=20, random_state=42)

top_n = 12
mdi_idx  = np.argsort(mdi)[::-1][:top_n]
perm_idx = np.argsort(perm.importances_mean)[::-1][:top_n]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=100)

axes[0].barh([X.columns[i] for i in mdi_idx[::-1]],
             mdi[mdi_idx[::-1]], color="#2a9d8f", edgecolor="white")
axes[0].set_title("MDI Feature Importance", fontweight="bold")

axes[1].barh([X.columns[i] for i in perm_idx[::-1]],
             perm.importances_mean[perm_idx[::-1]],
             xerr=perm.importances_std[perm_idx[::-1]],
             color="#1a472a", edgecolor="white")
axes[1].set_title("Permutation Importance", fontweight="bold")

plt.tight_layout(); plt.show()

<a id = "5"></a>
<center><h1 style = "background:#0d2b1a ;color:white;border:0;font-weight:bold">Evaluation of Model</h1></center>

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
results = [accuracy_score(y_test, pred), precision_score(y_test, pred),
           recall_score(y_test, pred), f1_score(y_test, pred),
           roc_auc_score(y_test, prob)]
pd.DataFrame({"Metric": ["Accuracy","Precision","Recall","F1","ROC-AUC"],
              "Score": results}).round(4)

In [ ]:
print(classification_report(y_test, pred, target_names=data.target_names))

cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(5, 4), dpi=100)
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=data.target_names, yticklabels=data.target_names)
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()